# CLass 6 Hansdon
- 🧱 Use .repartition() and .coalesce() to control data distribution across partitions
- ⚡ Implement .cache() and .persist() to avoid redundant computation and speed up transformations
- 📊 Detect and resolve skewed joins using techniques like salting and broadcast joins
- 💾 Write to Delta Tables and enable schema enforcement with ACID guarantees
- 🎯 Observe performance gains directly inside Databricks using Spark UI and job stages


# Actuall
- how to write data into deta lake or deta table
- how to run sql query over delta table
- mege into function in databricks and sql
- schema evolution 
- versioning in delta table or lake

In [0]:
data = [("Anurag", "IT", 50000), ("Stuti", "HR", 45000), ("Vaishnav", "IT", 60000)]
cols = ["Name", "Department", "Salary"]

df = spark.createDataFrame(data,cols)
df.display()

#### If by any chance the compute is disconnect so we will lose our data ..So tp permanently save this data into or file we can write this dtaa into different formats like "Delta" also we can save it as a file or save as table using "saveastable"

In [0]:
df.write.format("delta").mode("overwrite").save("FileStore/tables/deltaTable")

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("deltaTable")

In [0]:
%sql
-- even if the compute is of we can get the data from the table
select * from deltatable

In [0]:
%sql
describe history deltatable

In [0]:
%sql
update deltatable set Salary = 100000 where Name= 'Stuti';

#### The version that we are seeing here is bcz of delta table 

In [0]:
%sql
describe history deltatable

In [0]:
%sql
select * from deltatable version as of 0

In [0]:
spark.read.format("delta").option('timestampAsOf', '2026-02-03T00:54:24.000+00:00').table("deltaTable").display()


In [0]:

#MERGE INTO ➝ Update a Row

from pyspark.sql import Row

update_data = [Row(Name="Alice", Department="IT", Salary=52000)]
df_update = spark.createDataFrame(update_data)
df_update.createOrReplaceTempView("updates_view")


spark.sql("""
MERGE INTO deltaTable AS target
USING updates_view AS source
ON target.Name = source.Name
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
%sql
select * from deltatable;

# Schema Evolution


In [0]:
new = [("David", "Finance", 70000, 5)]
col= ["Name", "Department", "Salary", "Experience"]
df_new = spark.createDataFrame(new, col)

df_new.display()

# merge schema

In [0]:
df_new.write.format("delta").option("mergeSchema", True).mode("append")\
    .saveAsTable("deltaTable")

In [0]:
%sql
select * from deltatable

# see transaction logs

In [0]:
%sql
describe extended deltatable

# Practise Merge Into in Databricks

In [0]:
%sql
CREATE OR REPLACE TABLE customers_target (
  customer_id INT,
  name STRING,
  email STRING,
  city STRING,
  status STRING,
  updated_at TIMESTAMP
)
USING DELTA;

In [0]:
%sql
INSERT INTO customers_target VALUES
(1, 'Amit', 'amit@gmail.com', 'Mumbai', 'ACTIVE', '2024-01-01'),
(2, 'Neha', 'neha@gmail.com', 'Delhi', 'ACTIVE', '2024-01-05'),
(3, 'Ravi', 'ravi@gmail.com', 'Pune', 'ACTIVE', '2024-01-10');


In [0]:
%sql
select * from customers_target

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customers_source AS
SELECT * FROM VALUES
(2, 'Neha Sharma', 'neha_new@gmail.com', 'Delhi', false, '2024-02-01', 120),
(3, 'Ravi', null, 'Bangalore', false, '2024-01-08', 80),
(4, 'Kiran', 'kiran@gmail.com', 'Chennai', false, '2024-02-02', 50),
(5, 'Pooja', 'pooja@gmail.com', 'Mumbai', true, '2024-02-03', 0)
AS (
  customer_id,
  name,
  email,
  city,
  is_deleted,
  updated_at,
  loyalty_points
);


In [0]:
%sql
select * from customers_source

In [0]:
%sql
SET spark.databricks.delta.schema.autoMerge.enabled = true;


In [0]:
MERGE INTO customers_target t
USING customers_source s
ON t.customer_id = s.customer_id

WHEN MATCHED AND s.is_deleted = true THEN
  UPDATE SET
    status = 'INACTIVE',
    updated_at = s.updated_at

WHEN MATCHED AND s.updated_at > t.updated_at THEN
  UPDATE SET
    name = s.name,
    city = s.city,
    email = COALESCE(s.email, t.email),
    loyalty_points = s.loyalty_points,
    updated_at = s.updated_at

WHEN NOT MATCHED AND s.is_deleted = false THEN
  INSERT (
    customer_id,
    name,
    email,
    city,
    status,
    updated_at,
    loyalty_points
  )
  VALUES (
    s.customer_id,
    s.name,
    s.email,
    s.city,
    'ACTIVE',
    s.updated_at,
    s.loyalty_points
  );


In [0]:
%sql
MERGE INTO customers_target AS t
USING customers_source AS s
ON t.customer_id = s.customer_id

WHEN MATCHED AND s.is_deleted = True THEN
  UPDATE SET
    t.status = 'INACTIVE',
    t.updated_at = s.updated_at

WHEN MATCHED AND s.updated_at > t.updated_at THEN
  UPDATE SET
  t.name = s.name,
  t.city = s.city,
  t.email = COALESCE(s.email, t.email),
  t.loyalty_points = s.loyalty_points,
  t.updated_at = s.updated_at

WHEN NOT MATCHED AND s.is_deleted = False THEN
INSERT (
    customer_id,
    name,
    email,
    city,
    status,
    updated_at,
    loyalty_points
  )
  VALUES (
    s.customer_id,
    s.name,
    s.email,
    s.city,
    'ACTIVE',
    s.updated_at,
    s.loyalty_points
  );


In [0]:
data = [
    (1, "Asha", 40000),
    (2, "Rahul", 50000),
    (3, "Neeraj", 45000)
]
col = ["emp_id", "name", "salary"]
df = spark.createDataFrame(data, col)
df.write.format("delta").mode("overwrite").saveAsTable("emp_Table")

In [0]:
# Source data
source_data = [
    (2, "Rahul K", 52000),
    (3, "Neeraj", 45000),
    (4, "Pooja", 48000)
]
col = ['emp_id','name','salary']
df1 = spark.createDataFrame(source_data, col)
df1.createOrReplaceTempView("updates_view")



## Q-Write a MERGE INTO that:

-Updates name and salary when emp_id matches

-Inserts new employees when emp_id does not exist

In [0]:
spark.sql(
    '''
    MERGE INTO emp_Table as t 
    USING updates_view as s 
    ON t.emp_id = s.emp_id 
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    '''
)

In [0]:
%sql
select * from emp_table

## Q3️⃣

-Update salary only if source.salary > target.salary

👉 Which rows will update now?

In [0]:
spark.sql('''
MERGE INTO emp_Table t
USING updates_view s
ON t.emp_id = s.emp_id
WHEN MATCHED AND s.salary > t.salary THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
''')

## Q4️⃣

Modify the merge so that:

Name is always updated

Salary updates only if higher

In [0]:
%sql
MERGE INTO emp_Table t
USING updates_view s
ON t.emp_id = s.emp_id

WHEN MATCHED THEN
  UPDATE SET
    name = s.name,
    salary = CASE
               WHEN s.salary > t.salary THEN s.salary
               ELSE t.salary
             END

WHEN NOT MATCHED THEN
  INSERT (emp_id, name, salary)
  VALUES (s.emp_id, s.name, s.salary);
